# ARC live-body debug — whole-organism runs

Instrumented ARC play with the **live daemon body** (`require_live=True`, the
production gate). The daemon must be running:
`target/release/genesis-daemon` with `GENESIS_DATA_DIR=~/.local/share/genesis`.

Run top to bottom with the repo `.venv` as kernel (`python` ≥ 3.12, `arc` extra
installed). Nothing here writes to her scorecard unless `record=True`.

In [ ]:
# Setup — repo venv interpreter, mind attached to the live body socket.
import sys, collections, logging
sys.path.insert(0, '../python')
logging.basicConfig(level=logging.ERROR)

from mind import Mind
from parietal_lobe.arc import ArcPractice

SOCKET = '/home/mindy/.local/share/genesis/genesis.sock'
m = Mind(SOCKET)
m.client.connect()          # Mind builds the client; the caller owns connect()
m.enter_game_mode()
print('data_dir:', m.data_dir)

In [ ]:
# The body gate — must be ok/embodied before any measurement counts.
r = m.readiness(require_body=True)
print('ok:', r.ok, '| embodied:', r.embodied, '| phase:', r.phase)
print('wave:', r.dominant_wave, '| arousal:', round(r.arousal, 3),
      '| plasticity:', round(r.plasticity_gate, 3),
      '| adenosine:', round(r.adenosine, 3))
assert r.ok, r.reasons

In [ ]:
# Instrumented play — log the winning drive per decision and watch
# self-signature changes (transformers) plus affordance learning.
import basal_ganglia.world_policy as wp

srcs = collections.Counter()
sig_events = []
model = m._map_world
policy = m._world_policy

_orig_record = m._record_world_decision
def _record(token, choice, avail):
    srcs[(choice or {}).get('source', 'unknown')] += 1
    return _orig_record(token, choice, avail)
m._record_world_decision = _record

_orig_choose = wp.WorldPolicy.choose
_last_sig = [None]
def _choose(self, avail, *a, **k):
    sig = model.self_signature()
    if _last_sig[0] is not None and sig != _last_sig[0]:
        sig_events.append(('sig_change', model.tick))
    _last_sig[0] = sig
    return _orig_choose(self, avail, *a, **k)
wp.WorldPolicy.choose = _choose
print('instrumented')

In [ ]:
# One live run — pick an environment; ~200 steps ≈ the human norm.
# Available ids live under ~/.local/share/genesis/arc/environment_files/<game>/<version>
GAME = 'ls20-9607627b'   # e.g. 'ft09-0d8bbf25' (click-only), 'm0r0-492f87ba' (directional)
p = ArcPractice(m)   # require_live=True by default — the real gate
res = p.play(GAME, max_steps=200)
print('RESULT:', res.state, '| levels', res.levels_completed, '/', res.win_levels,
      '| steps', res.steps, '| resets', res.resets)
print('sources:', dict(srcs))
print('sig changes:', sig_events[:20])
if res.reasons: print('reasons:', res.reasons)

In [ ]:
# Affordance inventory at the end — what did the world teach this run?
affs = collections.Counter()
for e in model.entities.values():
    for a in getattr(e, 'affordances', {}) or {}:
        affs[a] += 1
print('affordances:', dict(affs))
print('known_free:', len(model.known_free), '| blocked:', len(model.known_blocked),
      '| refused:', len(model.refused_cells))
print('entities:', len(model.entities), '| action effects:',
      {a: dict(v) for a, v in model.action_effects.items()})

In [ ]:
# Sweep several environments — one result line each.
import os
envroot = '/home/mindy/.local/share/genesis/arc/environment_files'
for game in ['ft09', 'm0r0', 'ls20', 'vc33', 'lp85']:
    ver = sorted(os.listdir(f'{envroot}/{game}'))[0]
    srcs.clear()
    res = p.play(f'{game}-{ver}', max_steps=160)
    print(f'{game}: {res.state} lvls {res.levels_completed}/{res.win_levels}'
          f' steps {res.steps} resets {res.resets} | {dict(srcs)}')

In [ ]:
# Teardown — unwrap instrumentation, release the body.
wp.WorldPolicy.choose = _orig_choose
m._record_world_decision = _orig_record
m.exit_game_mode()
print('released')